# 0_data_processing.ipynb

**Purpose:**
1. Merge bibliometric, gender/discipline, and citation data; select each author's
   first publication in 2012-2016 and identify publications from their first
   three years ("early-career" publications).
2. Query the Altmetric API for tweet/mention counts on those publications.
3. Query the Twitter API for tweet text, timestamps, and tweeter identity for
   the original tweets referencing those publications.

**Manuscript:** Gender differences in online visibility of early-career researchers

**Required input file** (place in `./data/`):
- `paper_full_demo.csv` -- a de-identified demo sample (1,000 randomly
  sampled authors, with `author_id` and `doi` anonymized) of the real,
  pre-merged `paper_full` data, provided instead of the full real dataset
  for research-ethics/privacy reasons. For the actual analysis, replace this
  with your full, real `paper_full` file. Required columns: `author_id`,
  `gender`, `discipline`, `first_name`, `last_name`, `cohort`, `item_id`,
  `doi`, `pubyear`.

**Required credentials** (set as environment variables -- never hardcode keys):
`ALTMETRIC_API_KEY`, `TWITTER_BEARER_TOKEN`, `TWITTER_API_KEY`, `TWITTER_API_SECRET`,
`TWITTER_ACCESS_TOKEN`, `TWITTER_ACCESS_SECRET`

**Outputs** (written to `./result/`):
- `0_author_doi_3y_2012_2016_sample.csv`
- `1_doi_3y_2012_2016_tweet_sample.pkl`


In [ ]:
import os
import json
from pathlib import Path
from urllib.request import Request, urlopen
import ssl
import collections

import pandas as pd

# Run this notebook from the repository root so these relative paths resolve
# correctly (i.e. open Jupyter from the repo's top-level folder).
DATA_DIR = Path("data")
RESULTS_DIR = Path("result")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ENCODING = "utf-8"
pd.set_option("max_colwidth", 100)
pd.set_option("display.float_format", lambda x: "%.4f" % x)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 15)

### Credentials

Read from environment variables -- **never hardcode API keys in a notebook that
will be committed to a public repository.**

Set these before launching Jupyter, e.g. on macOS/Linux:
```bash
export ALTMETRIC_API_KEY=your_key_here
jupyter notebook
```


In [ ]:
ALTMETRIC_API_KEY = os.environ.get("ALTMETRIC_API_KEY")
TWITTER_BEARER_TOKEN = os.environ.get("TWITTER_BEARER_TOKEN")
TWITTER_API_KEY = os.environ.get("TWITTER_API_KEY")
TWITTER_API_SECRET = os.environ.get("TWITTER_API_SECRET")
TWITTER_ACCESS_TOKEN = os.environ.get("TWITTER_ACCESS_TOKEN")
TWITTER_ACCESS_SECRET = os.environ.get("TWITTER_ACCESS_SECRET")

if ALTMETRIC_API_KEY is None:
    raise EnvironmentError(
        "Set the ALTMETRIC_API_KEY environment variable before running "
        "(e.g. `export ALTMETRIC_API_KEY=...` on macOS/Linux, or "
        "`setx ALTMETRIC_API_KEY ...` on Windows)."
    )

### Load and merge base datasets

Merge author gender/discipline data with bibliometric records (1996-2021) and
citation counts, then select each author's first publication (2012-2016 cohorts)
and identify all of that author's publications within their first three career
years.

> **Fix applied:** the original notebook referenced an undefined variable
> `paper_index` when filtering the merged frame -- this uses `paper_full`,
> consistent with every other use of the merged data in this step.

In [ ]:
# CHANGE: paper_full is now read directly as one pre-merged file, rather than
# built from 3 separate raw sources (0_non_mover_gender_dis.csv, the
# bibliometric parquet, and the citation CSV) each time this notebook runs.
#
# NOTE: this reads "paper_full_demo.csv" -- a de-identified DEMO SAMPLE of
# the real paper_full data (1,000 randomly sampled authors, with author_id
# and doi anonymized), not the full real dataset. This demo file exists so
# the pipeline can be run, tested, and shared (e.g. for code review or a
# public repository) without exposing the real, identifiable author-level
# data, for research-ethics/privacy reasons. For the actual analysis, this
# path should point to your full, real paper_full file instead.
#
# Required columns: author_id, gender, discipline, first_name, last_name,
# cohort, item_id, doi, pubyear (citations/EID are not needed -- see note
# below).
paper_full = pd.read_csv(DATA_DIR / "paper_full_demo.csv", encoding=ENCODING)

required_cols = ["author_id", "gender", "discipline", "first_name", "last_name",
                  "cohort", "item_id", "doi", "pubyear"]

# Cohort = year of the author's first publication in the data
paper_full["cohort"] = paper_full.groupby("author_id")["pubyear"].transform("min")

missing = [c for c in required_cols if c not in paper_full.columns]
if missing:
    raise ValueError(f"paper_full_demo.csv is missing required columns: {missing}")

# NOTE: "citations" (from the original citation-count merge) is not used
# anywhere in this notebook -- if paper_full_demo includes it, that's fine
# (harmless extra column), but it isn't required here.
#
# NOTE, unresolved: "cohort" is used below (in the 3-year-window filter) but
# was never demonstrably created in the original 3-file-merge version either
# -- it must already exist as a column in your source data. Please confirm
# this is correct (e.g. cohort = the author's first-publication year) before
# treating this as final.

In [ ]:
# Select the first publication for each author (published between 2012 and 2016)
author_first = (
    paper_full.sort_values(by=["author_id", "pubyear"])
    .groupby("author_id")
    .head(1)
    .copy()
)

author_12_16 = author_first[
    (author_first["pubyear"] <= 2016) & (author_first["pubyear"] >= 2012)
].copy()
author_12_16 = author_12_16[
    author_12_16["gender"].notnull() & author_12_16["discipline"].notnull()
]

# Publications belonging to these early-career authors (fix: was `paper_index`)
paper_author_12_16 = paper_full[paper_full["author_id"].isin(author_12_16["author_id"])]

# NOTE on a fix: author_12_16 (one row per author, their first publication)
# and paper_author_12_16 (one row per publication) both have a "pubyear"
# column with different meanings -- merging without addressing this creates
# "pubyear_x"/"pubyear_y" instead of a clean "pubyear", and the filter below
# would raise a KeyError. Drop author_12_16's own pubyear (redundant with
# "cohort" for its one first-publication row) before merging, so the
# per-publication pubyear from paper_author_12_16 comes through cleanly.
#
# Also keep author_id/first_name/last_name/gender/discipline/cohort in the
# final output: 1_self_promotion.ipynb's very next step needs author_id,
# first_name, and last_name from this file, which the original notebook's
# `[["doi", "pubyear"]]` selection at the end dropped entirely.
# NOTE on a fix (extended): author_12_16 ALSO carries its own "doi" column
# (the author's first-publication DOI, inherited from paper_full via
# author_first), which collides with paper_author_12_16's per-publication
# "doi" the same way "pubyear" did -- drop both before merging.
author_12_16_paper = author_12_16.drop(columns=["pubyear", "doi"]).merge(
    paper_author_12_16[["author_id", "doi", "pubyear"]], on="author_id", how="left"
)
author_12_16_early_paper = author_12_16_paper[
    (author_12_16_paper["pubyear"] - author_12_16_paper["cohort"]) < 3
]
author_12_16_early_paper = author_12_16_early_paper[
    ["author_id", "first_name", "last_name", "gender", "discipline", "cohort", "doi", "pubyear"]
].drop_duplicates()

In [ ]:
author_12_16_early_paper.to_csv(RESULTS_DIR / "0_author_doi_3y_2012_2016.csv", index=False)
print(f"Wrote {len(author_12_16_early_paper)} rows to {RESULTS_DIR / '0_author_doi_3y_2012_2016.csv'}")

## Using Altmetric

Get the number of original tweets, the tweet-id list, and the tweeter-id list
for each publication's DOI.

In [ ]:
def fetch_first_time(response_json: dict, type_time: str) -> list:
    """Extract [year, month, day] for a given timestamp field in an Altmetric
    response, defaulting to [0, 0, 0] if the field is absent."""
    if type_time in response_json.get("citation", {}):
        date = response_json["citation"][type_time]
        return [int(date[0:4]), int(date[5:7]), int(date[8:10])]
    return [0, 0, 0]


def unique_keyvalue(lst, key1, key2):
    counts = collections.Counter(e[key1][key2] for e in lst if key1 in e and key2 in e[key1])
    return len(counts)


def unique_key_2_values(lst, key1, key2):
    return list({d[key1][key2] for d in lst if key1 in d and key2 in d[key1]})


def unique_key_1_values(lst, key1):
    return list({d[key1] for d in lst if key1 in d})


def key_2_values(lst, key1, key2):
    values = [d[key1][key2] for d in lst if key1 in d and key2 in d[key1]]
    return values, len(values)


def key_1_values(lst, key1):
    return [d[key1] for d in lst if key1 in d]

In [ ]:
def fetch_tweet_tweeter_nonunique(response_json: dict) -> list:
    """Extract the Original_Tweets count, Original_Tweet_list, Original_Tweeters
    count, and Original_Tweeter_list from an Altmetric API response. Adjust the
    key names below if Altmetric's schema differs for your API version."""
    posts = response_json.get("posts", {}).get("twitter", [])
    tweet_ids = [p.get("tweet_id") for p in posts]
    tweeter_ids = [p.get("tweeter_id") for p in posts]
    return [len(tweet_ids), tweet_ids, len(tweeter_ids), tweeter_ids]


def fetch_altmetric_record(doi: str, api_key: str, ssl_context) -> list:
    """Query the Altmetric Details Page API for one DOI's original-tweet info.
    Returns a flat list matching `header_list_y` below; on any request failure,
    returns just [doi] with everything else left for the caller to fill with
    NaN/empty defaults."""
    url_prefix = "https://api.altmetric.com/v1/fetch/doi/"
    result = [doi]
    try:
        req = Request(
            url=f"{url_prefix}{doi}?key={api_key}&post_types=original_tweets",
            headers={"User-Agent": "Mozilla/5.0"},
        )
        response = urlopen(req, context=ssl_context)
        response_json = json.load(response)
    except Exception:
        return result

    published_on = fetch_first_time(response_json, "published_on")
    first_seen_time = fetch_first_time(response_json, "first_seen_on")
    tweet_tweeter = fetch_tweet_tweeter_nonunique(response_json)
    result += published_on + first_seen_time + tweet_tweeter
    return result

### Altmetric collection

**Note:** this can take a long time and depends on live API access and rate
limits. Consider running it once and caching results, as later notebooks
(`1_self_promotion.ipynb`) read from the cached pickle file rather than
re-querying the API.

In [ ]:
ssl_context = ssl.create_default_context()
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE

header_list_y = [
    "doi", "Publish_year", "Publish_month", "Publish_day",
    "Al_year", "Al_month", "Al_day",
    "Original_Tweets", "Original_Tweet_list",
    "Original_Tweeters", "Original_Tweeter_list",
]

dois = author_12_16_early_paper["doi"].dropna().unique().tolist()
records = [fetch_altmetric_record(doi, ALTMETRIC_API_KEY, ssl_context) for doi in dois]

# Pad any short rows (failed requests) so DataFrame construction doesn't
# silently misalign columns.
records = [r + [None] * (len(header_list_y) - len(r)) for r in records]
out_put_all = pd.DataFrame(records, columns=header_list_y)
out_put_all.to_pickle(RESULTS_DIR / "1_doi_3y_2012_2016_tweet_sample.pkl")
print(f"Wrote Altmetric results to {RESULTS_DIR / '1_doi_3y_2012_2016_tweet_sample.pkl'}")

## Using Twitter API

2. Select the original tweets in the first month (`created_at`)
3. Identify self-promotion (`name`) -- see `1_self_promotion.ipynb`

**Note:** the batched Twitter-API tweet-fetching step (`get_time_name_df` in the
original notebook) depends on infrastructure not reproduced in this cleaned
version -- if you need to re-run this step, batch `Original_Tweet_list` IDs
through `client.get_tweets()` (tweepy) or the Twitter API v2 `tweets` endpoint
directly, using the credentials loaded above.